## Variables desctiption

In [53]:
import sympy as sp
from sympy import symbols, Quaternion, Matrix
from sympy.physics.vector import dynamicsymbols, init_vprinting
from sympy import Eq, Derivative
init_vprinting()

# Earth frame (inertial, ENU)

# Constants
g = dynamicsymbols('g')  # gravity
t = symbols('t')         # time
# Parameters
mass = symbols('m')          # mass of the MAV
b = symbols('b')             # drag coefficient

# State vector
q_w, q_x, q_y, q_z, v_x, v_y, v_z, p_x, p_y, p_z = dynamicsymbols('q_w, q_x, q_y, q_z, v_x, v_y, v_z, p_x, p_y, p_z')
w_x, w_y, w_z, tau = dynamicsymbols('w_x, w_y, w_z, tau')

q_vec_in_W = Matrix([q_w, q_x, q_y, q_z])
vel_in_W = Matrix([v_x, v_y, v_z])
pos_in_W = Matrix([p_x, p_y, p_z])

trust_vec_in_B = Matrix([0, 0, tau / mass])
grav_vec_in_W = Matrix([0, 0, -g])
drag_vec_in_W = -b * vel_in_W

quat_vec = Matrix([q_w, q_x, q_y, q_z])
quat = Quaternion(q_w, q_x, q_y, q_z)
quat_dot = quat.diff(t)
trust_in_W = Matrix(Quaternion.rotate_point(trust_vec_in_B, quat))



## System dynamics description

In [54]:
# Vel state transition equation
# The equation of motion for the velocity vector in the inertial frame
total_force = trust_in_W # + grav_vec_in_W + drag_vec_in_W

# Quaternion state transition equation
# The equation of motion for the quaternion vector in the inertial frame
# As we control quaternion directly with attitude control, this dynamics is quite simple, modeled as a first order lag
quat_rates = Quaternion(0.0, w_x, w_y, w_z)
quat_dyn = (quat_rates * quat) * 0.5

state_var = Matrix([q_vec_in_W, vel_in_W, pos_in_W, g])
dyn_fn = Matrix([quat_dyn.a, quat_dyn.b, quat_dyn.c, quat_dyn.d, total_force, vel_in_W, 0])
# State transition equation
eq = Eq(state_var.diff(t), dyn_fn)
display(dyn_fn)


⎡                                         -0.5⋅qₓ⋅wₓ - 0.5⋅q_y⋅w_y - 0.5⋅q_z⋅w
⎢                                                                             
⎢                                         0.5⋅q_w⋅wₓ - 0.5⋅q_y⋅w_z + 0.5⋅q_z⋅w
⎢                                                                             
⎢                                          0.5⋅q_w⋅w_y + 0.5⋅qₓ⋅w_z - 0.5⋅q_z⋅
⎢                                                                             
⎢                                          0.5⋅q_w⋅w_z - 0.5⋅qₓ⋅w_y + 0.5⋅q_y⋅
⎢                                                                             
⎢                                       2⋅q_w⋅q_y⋅τ                     2⋅qₓ⋅q
⎢                               ──────────────────────────── + ───────────────
⎢                                 ⎛   2     2      2      2⎞     ⎛   2     2  
⎢                               m⋅⎝q_w  + qₓ  + q_y  + q_z ⎠   m⋅⎝q_w  + qₓ  +
⎢                                                   

In [55]:
# Quaternion partial derivative normalization factor

q_norm = quat_vec.norm()
partial_q_expansion = (Matrix.ones(4, 4) - quat_vec * quat_vec.transpose() / q_norm**2) / q_norm
partial_q_expansion

⎡                      2                                                      
⎢                   q_w                                   q_w⋅qₓ              
⎢ 1 - ────────────────────────────────   1 - ──────────────────────────────── 
⎢          2       2        2        2            2       2        2        2 
⎢     │q_w│  + │qₓ│  + │q_y│  + │q_z│        │q_w│  + │qₓ│  + │q_y│  + │q_z│  
⎢─────────────────────────────────────  ───────────────────────────────────── 
⎢   __________________________________     __________________________________ 
⎢  ╱      2       2        2        2     ╱      2       2        2        2  
⎢╲╱  │q_w│  + │qₓ│  + │q_y│  + │q_z│    ╲╱  │q_w│  + │qₓ│  + │q_y│  + │q_z│   
⎢                                                                             
⎢                                                            2                
⎢                  q_w⋅qₓ                                  qₓ                 
⎢ 1 - ────────────────────────────────   1 - ───────

## System linearization

In [56]:
state_vars = [q_w, q_x, q_y, q_z, v_x, v_y, v_z, p_x, p_y, p_z, g]
control_vars = [w_x, w_y, w_z, tau]

# Jacobian of the state transition equation
A = sp.Matrix.zeros(len(state_vars), len(state_vars))
for i in range(len(state_vars)):
    for j in range(len(state_vars)):
        A[i, j] = Derivative(eq.rhs[i], state_vars[j]).doit()
A[4:, 0:4] = A[4:, 0:4]
display(A)

B = sp.Matrix.zeros(len(state_vars), len(control_vars))
for i in range(len(state_vars)):
    for j in range(len(control_vars)):
        B[i, j] = Derivative(eq.rhs[i], control_vars[j]).doit()
display(B)


⎡                                                                             
⎢                                                                             
⎢                                                                           0.
⎢                                                                             
⎢                                                                          0.5
⎢                                                                             
⎢                                                                          0.5
⎢                                                                             
⎢                                                                             
⎢                                          2⋅q_y⋅τ                       4⋅q_w
⎢                                ──────────────────────────── - ──────────────
⎢                                  ⎛   2     2      2      2⎞                 
⎢                                m⋅⎝q_w  + qₓ  + q_y

⎡-0.5⋅qₓ   -0.5⋅q_y  -0.5⋅q_z                                                 
⎢                                                                             
⎢0.5⋅q_w   0.5⋅q_z   -0.5⋅q_y                                                 
⎢                                                                             
⎢-0.5⋅q_z  0.5⋅q_w    0.5⋅qₓ                                                  
⎢                                                                             
⎢0.5⋅q_y   -0.5⋅qₓ   0.5⋅q_w                                                  
⎢                                                                             
⎢                                                                      2⋅q_w⋅q
⎢   0         0         0                                     ────────────────
⎢                                                               ⎛   2     2   
⎢                                                             m⋅⎝q_w  + qₓ  + 
⎢                                                   

In [57]:
# Separated control synthesis

A_quat = A[0:4, 0:4]
B_quat = B[0:4, 0:4]

A_vel = A[4:7, 4:7]
B_vel = B[4:7, 4:7]

sp.Add(sp.MatMul(A_quat, sp.Matrix(state_vars[0:4]), evaluate=False), sp.MatMul(B_quat, sp.Matrix(control_vars[0:4]), evaluate=False), evaluate=False)

⎡   0     -0.5⋅wₓ   -0.5⋅w_y  -0.5⋅w_z⎤ ⎡q_w⎤   ⎡-0.5⋅qₓ   -0.5⋅q_y  -0.5⋅q_z 
⎢                                     ⎥ ⎢   ⎥   ⎢                             
⎢0.5⋅wₓ      0      -0.5⋅w_z  0.5⋅w_y ⎥ ⎢qₓ ⎥   ⎢0.5⋅q_w   0.5⋅q_z   -0.5⋅q_y 
⎢                                     ⎥⋅⎢   ⎥ + ⎢                             
⎢0.5⋅w_y  0.5⋅w_z      0      -0.5⋅wₓ ⎥ ⎢q_y⎥   ⎢-0.5⋅q_z  0.5⋅q_w    0.5⋅qₓ  
⎢                                     ⎥ ⎢   ⎥   ⎢                             
⎣0.5⋅w_z  -0.5⋅w_y   0.5⋅wₓ      0    ⎦ ⎣q_z⎦   ⎣0.5⋅q_y   -0.5⋅qₓ   0.5⋅q_w  

 0⎤ ⎡wₓ ⎤
  ⎥ ⎢   ⎥
 0⎥ ⎢w_y⎥
  ⎥⋅⎢   ⎥
 0⎥ ⎢w_z⎥
  ⎥ ⎢   ⎥
 0⎦ ⎣ τ ⎦

In [58]:
sp.Add(sp.MatMul(A_vel, sp.Matrix(state_vars[4:7]), evaluate=False), sp.MatMul(B_vel, sp.Matrix(control_vars[4:5]), evaluate=False), evaluate=False)

        ⎡0  0  0⎤ ⎡vₓ ⎤
        ⎢       ⎥ ⎢   ⎥
[]⋅[] + ⎢0  0  0⎥⋅⎢v_y⎥
        ⎢       ⎥ ⎢   ⎥
        ⎣0  0  0⎦ ⎣v_z⎦